# SynthFin-AML V9.1 Benchmark Tutorial

This notebook demonstrates how to load the SynthFin-AML V9.1 dataset and evaluate baseline models for anti-money laundering (AML) detection. We compare a standard gradient boosting approach (LightGBM) against a graph neural network (Temporal EdgeSAGE).

**Workflow:**
1. Load the V9.1 dataset from HuggingFace.
2. Train a baseline LightGBM model on tabular features.
3. Train a Temporal EdgeSAGE model on the transaction graph.
4. Compare model performance using PR-AUC, Precision@Top-500, and inference Latency.

**Why Tabular Models Fail on V9.1**
In this release, we removed trivial amount leakage—such as exact, round-number illicit transfers—that previously allowed tabular models to overfit. We embedded realistic structuring typologies, where fraudulent actors use high-frequency fan-out/fan-in patterns to dynamically split their transfers. Viewed in isolation, these look identical to normal P2P or escrow economic activity, causing standard tabular models to fail. Graph-based models like EdgeSAGE succeed by capturing the temporal and structural network patterns.


In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import precision_recall_curve, auc, classification_report
import lightgbm as lgb
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import SAGEConv

# Formatting
pd.set_option('display.float_format', lambda x: '%.3f' % x)

def calc_biz_metrics(y_true, y_prob, name, t_infer, num_rows):
    if isinstance(y_true, pd.Series):
        y_true = y_true.values
    idx = np.argsort(y_prob)[::-1]
    y_true_sorted = y_true[idx]
    prec_500 = y_true_sorted[:500].sum() / 500.0 if len(y_true_sorted) >= 500 else y_true_sorted.mean()
    lat_ms = (t_infer / max(1, num_rows)) * 1000
    print(f"[{name}] Precision@Top-500: {prec_500:.2%} | Latency: {lat_ms:.2f}ms / 1k tx")


## 1. Load the Dataset
We load the `synthfin-small` dataset (10k nodes, ~125k transactions).


In [ ]:
print("Loading edges...")
# In a real environment, you'd download this from HuggingFace
edges_df = pd.read_csv('../datasets/synthfin-small_edges.csv')

print(f"Loaded {len(edges_df):,} transactions.")
print(f"Fraud ratio: {edges_df['is_fraud'].mean():.2%}")

# Sort by time to strictly prevent future-data leakage
edges_df = edges_df.sort_values('timestamp').reset_index(drop=True)
edges_df.head()


## 2. Temporal Split (Out-Of-Time)
In AML, we must predict future transactions based on past data. We will use the first 80% of transactions for training and the last 20% for testing.


In [ ]:
train_size = int(len(edges_df) * 0.8)
train_df = edges_df.iloc[:train_size].copy()
test_df = edges_df.iloc[train_size:].copy()

print(f"Train edges: {len(train_df):,} | Test edges: {len(test_df):,}")
print(f"Train fraud ratio: {train_df['is_fraud'].mean():.2%} | Test fraud ratio: {test_df['is_fraud'].mean():.2%}")


## 3. LightGBM Baseline
We create simple rolling features for the tabular model.


In [ ]:
def build_tabular_features(df):
    features = df[['amount']].copy()
    features['amount_log'] = np.log1p(df['amount'])
    features['hour'] = (df['timestamp'] // 3600) % 24
    
    # Adding simple rolling degree count (proxy for behavior)
    df['dummy'] = 1
    # Very crude approximation for tabular baseline to run fast
    return features

X_train = build_tabular_features(train_df)
y_train = train_df['is_fraud']
X_test = build_tabular_features(test_df)
y_test = test_df['is_fraud']

model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)

# Training
t0 = time.time()
model.fit(X_train, y_train)
t_train_lgb = time.time() - t0

# Inference
t0 = time.time()
preds_lgb = model.predict_proba(X_test)[:, 1]
t_infer_lgb = time.time() - t0

precision, recall, _ = precision_recall_curve(y_test, preds_lgb)
pr_auc_lgb = auc(recall, precision)

print(f"LightGBM PR-AUC: {pr_auc_lgb:.4f}")
print(f"Train Time: {t_train_lgb:.3f}s | Infer Time: {t_infer_lgb:.3f}s (for {len(X_test):,} rows)")
calc_biz_metrics(y_test, preds_lgb, "LightGBM", t_infer_lgb, len(X_test))


## 4. Temporal EdgeSAGE (GNN)
Now we construct a PyTorch Geometric (PyG) graph and train a Temporal EdgeSAGE model.


In [ ]:
# Node indices mapping
unique_nodes = pd.concat([edges_df['source'], edges_df['target']]).unique()
node_mapping = {n: i for i, n in enumerate(unique_nodes)}

src_idx = torch.tensor([node_mapping[n] for n in edges_df['source']], dtype=torch.long)
tgt_idx = torch.tensor([node_mapping[n] for n in edges_df['target']], dtype=torch.long)
edge_index = torch.stack([src_idx, tgt_idx], dim=0)

edge_attr = torch.tensor(np.log1p(edges_df['amount'].values), dtype=torch.float).view(-1, 1)
edge_time = torch.tensor(edges_df['timestamp'].values, dtype=torch.float)
edge_label = torch.tensor(edges_df['is_fraud'].values, dtype=torch.float)

# Simple node features (just degree or constant)
x = torch.ones((len(unique_nodes), 1), dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, time=edge_time)

# Masking for temporal training
train_mask = torch.zeros(len(edges_df), dtype=torch.bool)
train_mask[:train_size] = True
test_mask = ~train_mask

# Train loader (samples temporal neighbors)
train_loader = LinkNeighborLoader(
    data,
    num_neighbors=[10, 5],
    time_attr='time',
    edge_label_index=data.edge_index[:, train_mask],
    edge_label=edge_label[train_mask],
    edge_label_time=edge_time[train_mask],
    batch_size=1024,
    shuffle=True,
)

# Test loader
test_loader = LinkNeighborLoader(
    data,
    num_neighbors=[10, 5],
    time_attr='time',
    edge_label_index=data.edge_index[:, test_mask],
    edge_label=edge_label[test_mask],
    edge_label_time=edge_time[test_mask],
    batch_size=4096,
    shuffle=False,
)


In [ ]:
class TemporalEdgeSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        # edge classifier: src_emb || tgt_emb || edge_attr
        self.edge_mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels * 2 + 1, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index, edge_attr):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index).relu()
        return h

    def predict_edge(self, h, edge_label_index, edge_attr):
        src_h = h[edge_label_index[0]]
        tgt_h = h[edge_label_index[1]]
        edge_rep = torch.cat([src_h, tgt_h, edge_attr], dim=-1)
        return self.edge_mlp(edge_rep).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_gnn = TemporalEdgeSAGE(in_channels=1, hidden_channels=32, out_channels=1).to(device)
optimizer = torch.optim.Adam(model_gnn.parameters(), lr=0.005)
criterion = torch.nn.BCEWithLogitsLoss()


In [ ]:
print("Training GNN for 3 epochs...")
t0 = time.time()
model_gnn.train()
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        h = model_gnn(batch.x, batch.edge_index, batch.edge_attr)
        
        # In mini-batch, edge_label_index represents the edges we are predicting
        # But we need their corresponding attributes.
        # LinkNeighborLoader provides input_id for the edges in the batch
        pred = model_gnn.predict_edge(h, batch.edge_label_index, batch.edge_attr[:batch.edge_label_index.size(1)])
        loss = criterion(pred, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")
t_train_gnn = time.time() - t0


In [ ]:
print("Evaluating GNN...")
model_gnn.eval()
preds_gnn = []
labels_gnn = []

t0 = time.time()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        h = model_gnn(batch.x, batch.edge_index, batch.edge_attr)
        pred = model_gnn.predict_edge(h, batch.edge_label_index, batch.edge_attr[:batch.edge_label_index.size(1)])
        preds_gnn.append(torch.sigmoid(pred).cpu())
        labels_gnn.append(batch.edge_label.cpu())
t_infer_gnn = time.time() - t0

preds_gnn = torch.cat(preds_gnn).numpy()
labels_gnn = torch.cat(labels_gnn).numpy()

precision_gnn, recall_gnn, _ = precision_recall_curve(labels_gnn, preds_gnn)
pr_auc_gnn = auc(recall_gnn, precision_gnn)

print(f"Temporal EdgeSAGE PR-AUC: {pr_auc_gnn:.4f} (Lift: {pr_auc_gnn - pr_auc_lgb:+.4f})")
print(f"Train Time: {t_train_gnn:.3f}s | Infer Time: {t_infer_gnn:.3f}s")
calc_biz_metrics(labels_gnn, preds_gnn, "Temporal EdgeSAGE", t_infer_gnn, len(labels_gnn))


## 5. Latency vs Accuracy (The Hybrid Proposal)
Notice the difference in inference times!
LightGBM inference takes ~0.05s, while the GNN takes significantly longer (~2-5s) because it has to sample topological neighbors.

**Next steps for production:** 
Use the GNN to extract node embeddings (`h`), and feed those as tabular features into LightGBM to get both the high accuracy of graph features and the ultra-low latency of tabular inference. Let's do that now!


In [ ]:
print("Extracting GNN embeddings...")
model_gnn.eval()

# We need the full graph embeddings
with torch.no_grad():
    # Pass all nodes and edges to get the final node embeddings
    h_full = model_gnn(data.x.to(device), data.edge_index.to(device), data.edge_attr.to(device))

# Move back to CPU
h_full_np = h_full.cpu().numpy()

# Add to the existing train/test DataFrames
# Each edge has a 'source' and 'target' which map to node_mapping
train_src_emb = h_full_np[[node_mapping[n] for n in train_df['source']]]
train_tgt_emb = h_full_np[[node_mapping[n] for n in train_df['target']]]

test_src_emb = h_full_np[[node_mapping[n] for n in test_df['source']]]
test_tgt_emb = h_full_np[[node_mapping[n] for n in test_df['target']]]

# We will concatenate the tabular features with the embeddings
X_train_hybrid = np.hstack([X_train.values, train_src_emb, train_tgt_emb])
X_test_hybrid = np.hstack([X_test.values, test_src_emb, test_tgt_emb])

print(f"Hybrid feature shape: {X_train_hybrid.shape}")


## 6. Train Hybrid LightGBM Model


In [ ]:
model_hybrid = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)

# Training Hybrid
t0 = time.time()
model_hybrid.fit(X_train_hybrid, y_train)
t_train_hybrid = time.time() - t0

# Inference Hybrid
t0 = time.time()
preds_hybrid = model_hybrid.predict_proba(X_test_hybrid)[:, 1]
t_infer_hybrid = time.time() - t0

precision_hybrid, recall_hybrid, _ = precision_recall_curve(y_test, preds_hybrid)
pr_auc_hybrid = auc(recall_hybrid, precision_hybrid)

print(f"LightGBM (Tabular) PR-AUC: {pr_auc_lgb:.4f}")
print(f"Temporal EdgeSAGE PR-AUC: {pr_auc_gnn:.4f}")
print(f"Hybrid (GNN+LGBM) PR-AUC: {pr_auc_hybrid:.4f}")
print("-" * 30)
print(f"Hybrid Infer Time: {t_infer_hybrid:.3f}s (vs Tabular {t_infer_lgb:.3f}s vs GNN {t_infer_gnn:.3f}s)")
calc_biz_metrics(y_test, preds_hybrid, "Hybrid (GNN+LGBM)", t_infer_hybrid, len(X_test_hybrid))
